In [ ]:
import os
import json
import base64
from io import BytesIO
from dataclasses import dataclass
from typing import Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None



In [ ]:

# ----------------------------
# 1) Daten erzeugen
# ----------------------------
def generate_random_revenue_data(seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    months = ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"]
    channels = ["App", "Website", "Beratungsgespraech"]
    categories = ["Fondssparen", "Versicherung", "Immobilienvermittlung"]

    rows = []
    for month_idx, month in enumerate(months, start=1):
        seasonality = 1.0 + 0.15 * np.sin((month_idx / 12) * 2 * np.pi)
        for ch in channels:
            ch_factor = {"App": 1.10, "Website": 1.00, "Beratungsgespraech": 0.95}[ch]
            for cat in categories:
                cat_factor = {
                    "Fondssparen": 1.05,
                    "Versicherung": 0.90,
                    "Immobilienvermittlung": 1.20
                }[cat]
                base = 80000
                noise = rng.normal(0, 9000)
                revenue = max(5000, base * seasonality * ch_factor * cat_factor + noise)

                rows.append({
                    "month": month,
                    "month_idx": month_idx,
                    "channel": ch,
                    "product_category": cat,
                    "revenue_eur": round(float(revenue), 2)
                })

    return pd.DataFrame(rows)


# ----------------------------
# 2) LLM-Helfer
# ----------------------------
def _get_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key or OpenAI is None:
        return None
    return OpenAI(api_key=api_key)


def _b64_png(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def _safe_json_loads(text: str) -> Dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        # grober fence-strip
        lines = text.splitlines()
        if lines and lines[0].lower().startswith("json"):
            lines = lines[1:]
        text = "\n".join(lines)
    return json.loads(text)


def call_llm_json(system_prompt: str, user_prompt: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
    client = _get_client()
    if client is None:
        return fallback

    try:
        resp = client.responses.create(
            model="gpt-4.1",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        txt = resp.output_text
        return _safe_json_loads(txt)
    except Exception:
        return fallback


def review_chart_with_vision(chart_path: str, summary_text: str) -> Dict[str, Any]:
    fallback = {
        "score_readability": 6,
        "score_insight": 6,
        "score_layout": 6,
        "issues": [
            "Beschriftungen könnten klarer sein.",
            "Legende und Titel könnten präziser sein."
        ],
        "improvements": [
            "Y-Achse als EUR (Tsd.) formatieren.",
            "Titel + Untertitel mit Zeitraum ergänzen.",
            "Farben konsistent und kontrastreich wählen."
        ]
    }

    client = _get_client()
    if client is None:
        return fallback

    try:
        image_b64 = _b64_png(chart_path)
        prompt = f"""
Bewerte die Grafik kritisch. Kontext:
{summary_text}

Gib NUR JSON zurück:
{{
  "score_readability": 1-10,
  "score_insight": 1-10,
  "score_layout": 1-10,
  "issues": ["..."],
  "improvements": ["..."]
}}
"""
        resp = client.responses.create(
            model="gpt-4.1",
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{image_b64}"}
                ]
            }],
        )
        return _safe_json_loads(resp.output_text)
    except Exception:
        return fallback


# ----------------------------
# 3) Rendering über Spec
# ----------------------------
DEFAULT_SPEC = {
    "title": "Monatliche Erträge nach Kanal und Produktkategorie",
    "subtitle": "Kanäle: App, Website, Beratungsgespräch | Jahr: 12 Monate",
    "palette": {
        "App": "#1f77b4",
        "Website": "#2ca02c",
        "Beratungsgespraech": "#ff7f0e"
    },
    "show_value_labels": False,
    "legend_loc": "upper left",
    "rotate_xticks": 0
}


def render_chart(df: pd.DataFrame, spec: Dict[str, Any], out_path: str) -> None:
    spec = {**DEFAULT_SPEC, **spec}
    categories = ["Fondssparen", "Versicherung", "Immobilienvermittlung"]
    channels = ["App", "Website", "Beratungsgespraech"]
    months = ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    width = 0.25
    x = np.arange(len(months))

    for i, cat in enumerate(categories):
        ax = axes[i]
        dcat = df[df["product_category"] == cat].copy()
        dcat["month"] = pd.Categorical(dcat["month"], categories=months, ordered=True)
        dcat = dcat.sort_values("month")

        for j, ch in enumerate(channels):
            vals = (
                dcat[dcat["channel"] == ch]
                .groupby("month", observed=False)["revenue_eur"]
                .sum()
                .reindex(months)
                .values
            )
            bars = ax.bar(
                x + (j - 1) * width,
                vals / 1000.0,
                width=width,
                label=ch,
                color=spec["palette"].get(ch, None),
                alpha=0.9
            )
            if spec["show_value_labels"]:
                for b in bars:
                    h = b.get_height()
                    ax.text(b.get_x() + b.get_width()/2, h + 1, f"{h:.0f}", ha="center", va="bottom", fontsize=7)

        ax.set_title(cat)
        ax.set_xticks(x)
        ax.set_xticklabels(months, rotation=spec["rotate_xticks"])
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    axes[0].set_ylabel("Ertrag (Tsd. EUR)")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc=spec["legend_loc"], ncol=3, frameon=False, bbox_to_anchor=(0.08, 1.02))
    fig.suptitle(spec["title"], fontsize=14, y=1.06)
    fig.text(0.5, 1.01, spec["subtitle"], ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# ----------------------------
# 4) Agentic Workflow
# ----------------------------
def build_summary(df: pd.DataFrame) -> str:
    agg = df.groupby(["channel", "product_category"], as_index=False)["revenue_eur"].sum()
    top = agg.sort_values("revenue_eur", ascending=False).head(5)
    return (
        "Datensatz: Monatliche Erträge (12 Monate) für 3 Kanäle x 3 Produktkategorien.\n"
        f"Gesamt-Ertrag: {df['revenue_eur'].sum():,.0f} EUR\n"
        f"Top-Kombinationen:\n{top.to_string(index=False)}"
    )


def creator_agent(summary: str) -> Dict[str, Any]:
    fallback = {
        "title": "Erträge pro Monat nach Kanal und Produktkategorie",
        "subtitle": "Zufallsdaten, 12 Monate",
        "show_value_labels": False,
        "legend_loc": "upper left",
        "rotate_xticks": 0
    }
    system = "Du bist Data-Visualization-Agent. Gib nur valides JSON zurück."
    user = f"""
Entwirf eine Spec für ein klares Business-Chart.
Kontext:
{summary}

Gib nur JSON mit Feldern:
title, subtitle, show_value_labels, legend_loc, rotate_xticks
"""
    return call_llm_json(system, user, fallback)


def improver_agent(original_spec: Dict[str, Any], review: Dict[str, Any]) -> Dict[str, Any]:
    fallback = {
        **original_spec,
        "subtitle": (original_spec.get("subtitle", "") + " | Optimiert nach Feedback").strip(" |"),
        "rotate_xticks": 30,
        "legend_loc": "upper center",
        "show_value_labels": False
    }

    system = "Du verbesserst Visualisierungen. Gib nur valides JSON zurück."
    user = f"""
Original-Spec:
{json.dumps(original_spec, ensure_ascii=False)}

Review:
{json.dumps(review, ensure_ascii=False)}

Gib eine verbesserte JSON-Spec zurück mit Feldern:
title, subtitle, show_value_labels, legend_loc, rotate_xticks
"""
    return call_llm_json(system, user, fallback)


def main():
    df = generate_random_revenue_data(seed=42)
    summary = build_summary(df)

    # Agent 1: Creator
    spec_v1 = creator_agent(summary)
    render_chart(df, spec_v1, "revenue_chart_v1.png")

    # Agent 2: Reviewer
    review = review_chart_with_vision("revenue_chart_v1.png", summary)

    # Agent 3: Improver
    spec_v2 = improver_agent(spec_v1, review)
    render_chart(df, spec_v2, "revenue_chart_v2.png")

    # Ausgabe
    print("Workflow fertig.")
    print("- Erstversion: revenue_chart_v1.png")
    print("- Review:", json.dumps(review, ensure_ascii=False, indent=2))
    print("- Verbesserte Version: revenue_chart_v2.png")
    print("- Finale Spec:", json.dumps(spec_v2, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()